# Seed variance table (the ruler)

RMSE / CSI@0.05 / CSI@0.30 for each seed of the best-sweep config (seeds 666, 1, 200, 450, 800).

The mean ± spread of these values is the reference interval to judge every later change.

In [ ]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # force CPU
os.environ['OMP_NUM_THREADS'] = '1'        # single-thread: this machine's torch crashes otherwise
os.environ['MKL_NUM_THREADS'] = '1'

# notebook lives in utils/, repo root is one level up
REPO_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'utils' else os.getcwd()
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

import torch
torch.set_num_threads(1)
torch.set_num_interop_threads(1)
import wandb
import numpy as np
import pandas as pd

from utils.load import read_config
from utils.miscellaneous import get_model, fix_dict_in_config
from utils.dataset import create_model_dataset, get_temporal_test_dataset_parameters, to_temporal_dataset
from utils.visualization import PlotRollout
from training.train import LightningTrainer

torch.backends.cudnn.deterministic = True
torch.set_float32_matmul_precision('high')

print(REPO_ROOT)

## Config and test dataset (loaded once, shared by all checkpoints)

In [ ]:
CONFIG = 'config_best_sweep.yaml'

cfg = read_config(CONFIG)
wandb.init(mode='disabled', project='mswe-gnn', config=cfg)
fix_dict_in_config(wandb)
config = wandb.config
device = torch.device('cpu')

_, _, test_dataset, scalers = create_model_dataset(
    scalers=config.scalers, device=device,
    **config.dataset_parameters,
    **config.selected_node_features,
    **config.selected_edge_features
)
temporal_test_dataset_parameters = get_temporal_test_dataset_parameters(
    config, config.temporal_dataset_parameters
)
temporal_test_dataset = to_temporal_dataset(
    test_dataset, rollout_steps=-1, **temporal_test_dataset_parameters
)
num_node_features = temporal_test_dataset[0].x.size(-1)
num_edge_features = temporal_test_dataset[0].edge_attr.size(-1)
print('WD shape:', test_dataset[0].WD.shape)

## Checkpoints

Each run saved two checkpoints: `<name>.h5` (best val_loss epoch) and `<name>_bestCSI.h5` (best val_CSI_005 epoch).

Fix the seed-666 paths if the original best-sweep run used a different output name.

In [ ]:
CHECKPOINTS = {
    666: {'best_val_loss': 'results/best_sweep_new_gt.h5',
          'best_CSI':      'results/best_sweep_new_gt_bestCSI.h5'},
    1:   {'best_val_loss': 'results/best_sweep_new_gt_seed1.h5',
          'best_CSI':      'results/best_sweep_new_gt_seed1_bestCSI.h5'},
    200: {'best_val_loss': 'results/best_sweep_new_gt_seed200.h5',
          'best_CSI':      'results/best_sweep_new_gt_seed200_bestCSI.h5'},
    450: {'best_val_loss': 'results/best_sweep_seed450.h5',
          'best_CSI':      'results/best_sweep_seed450_bestCSI.h5'},
    800: {'best_val_loss': 'results/best_sweep_seed800.h5',
          'best_CSI':      'results/best_sweep_seed800_bestCSI.h5'},
}

# quick check which files are there
for seed, ckpts in CHECKPOINTS.items():
    for label, path in ckpts.items():
        print(f"seed {seed:3d} {label:14s} {'OK' if os.path.exists(path) else 'MISSING'}  {path}")

## Evaluation function (same logic as run_inference.py)

In [ ]:
def evaluate_checkpoint(ckpt_path):
    '''Load one checkpoint, run the full rollout, return epoch + metrics.'''
    model_parameters = dict(config.models)
    model_type = model_parameters.pop('model_type')
    if model_type == 'MSGNN':
        model_parameters['num_scales'] = test_dataset[0].mesh.num_meshes

    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

    model = get_model(model_type)(
        num_node_features=num_node_features,
        num_edge_features=num_edge_features,
        previous_t=temporal_test_dataset_parameters['previous_t'],
        device=device,
        **model_parameters
    ).to(device)

    plmodule = LightningTrainer.load_from_checkpoint(
        ckpt_path, map_location=device,
        model=model,
        lr_info=config['lr_info'],
        trainer_options=config.trainer_options,
        temporal_test_dataset_parameters=temporal_test_dataset_parameters
    )
    model = plmodule.model.to(device)
    model.eval()

    plot_rollout = PlotRollout(
        model, test_dataset[0], scalers=scalers,
        warmup_steps=3,
        **temporal_test_dataset_parameters
    )
    rollout_loss = plot_rollout._get_rollout_loss(type_loss='RMSE')
    loss_mean = rollout_loss.mean(0)
    rmse_wd = loss_mean[0].item() if loss_mean.dim() > 0 else loss_mean.item()
    csi_005 = plot_rollout._get_CSI(water_threshold=0.05).nanmean().item()
    csi_03  = plot_rollout._get_CSI(water_threshold=0.30).nanmean().item()

    return ckpt.get('epoch', -1), rmse_wd, csi_005, csi_03

## Evaluate all seeds

(each full rollout takes a few minutes on CPU, so ~10-20 min for 10 checkpoints)

In [ ]:
rows = []
for seed, ckpts in CHECKPOINTS.items():
    for label, path in ckpts.items():
        if not os.path.exists(path):
            print(f'skipped (missing): seed {seed} {label}')
            continue
        print(f'evaluating seed {seed} ({label})...')
        epoch, rmse_wd, csi005, csi03 = evaluate_checkpoint(path)
        rows.append(dict(seed=seed, ckpt=label, epoch=epoch,
                         RMSE_WD=rmse_wd, CSI_005=csi005, CSI_03=csi03))
        print(f'  epoch {epoch}: RMSE_WD={rmse_wd:.4f} m, CSI@0.05={csi005:.4f}, CSI@0.30={csi03:.4f}')

df = pd.DataFrame(rows)
df

## The ruler: mean ± spread per checkpoint type

The `best_CSI` block is the headline (judge metric = CSI@0.05); `best_val_loss` is supporting.

The best epochs are also plateau evidence for Linea 1 (where training stops improving).

In [ ]:
for label in ['best_CSI', 'best_val_loss']:
    sub = df[df.ckpt == label]
    if len(sub) < 2:
        continue
    print(f'=== RULER ({label}, n={len(sub)}) ===')
    for metric in ['RMSE_WD', 'CSI_005', 'CSI_03']:
        vals = sub[metric].values
        print(f'  {metric:8s}: mean={vals.mean():.4f} +- {vals.std(ddof=1):.4f}   range [{vals.min():.4f}, {vals.max():.4f}]')
    print(f'  best epochs: {sub.epoch.tolist()}')
    print()

In [ ]:
# save for the thesis appendix
df.to_csv('results/seed_table.csv', index=False)
print('saved results/seed_table.csv')

# markdown version to paste in the thesis / email
print(df.to_markdown(index=False, floatfmt='.4f'))